## SHAP BEESWARM — ONE-CELL MASTER CODE

✔ Binary classification

✔ Tree-based model

✔ Beeswarm (main)

✔ Bar + class-wise variants

✔ Clean & reproducible

In [ ]:
# ============================================================
# SHAP Beeswarm Plot — FULL ONE-CELL SCRIPT
# ============================================================

# (1) Install SHAP (run once if needed)
!pip -q install shap

# (2) Imports
import numpy as np
import pandas as pd
import shap
import matplotlib.pyplot as plt

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier

np.random.seed(42)

# ------------------------------------------------------------
# (3) Dataset
# ------------------------------------------------------------
data = load_breast_cancer()
X = pd.DataFrame(data.data, columns=data.feature_names)
y = data.target

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# ------------------------------------------------------------
# (4) Train Model
# ------------------------------------------------------------
model = RandomForestClassifier(
    n_estimators=300,
    max_depth=6,
    random_state=42,
    n_jobs=-1
)
model.fit(X_train, y_train)

# ------------------------------------------------------------
# (5) SHAP Explainer
# ------------------------------------------------------------
explainer = shap.TreeExplainer(model)
shap_values = explainer.shap_values(X_test)

# For binary classification:
# shap_values[1] corresponds to the positive class

# ------------------------------------------------------------
# (6) SHAP Beeswarm Plot (MAIN PLOT)
# ------------------------------------------------------------
shap.summary_plot(
    shap_values[1],
    X_test,
    plot_type="dot",
    max_display=15
)

# ------------------------------------------------------------
# (7) SHAP Beeswarm (Smaller subset – clarity)
# ------------------------------------------------------------
shap.summary_plot(
    shap_values[1],
    X_test.sample(200, random_state=42),
    plot_type="dot",
    max_display=15
)

# ------------------------------------------------------------
# (8) SHAP Bar Plot (Global importance only)
# ------------------------------------------------------------
shap.summary_plot(
    shap_values[1],
    X_test,
    plot_type="bar",
    max_display=15
)

# ------------------------------------------------------------
# (9) Class-wise Beeswarm (Optional but powerful)
# ------------------------------------------------------------
shap.summary_plot(
    shap_values,
    X_test,
    class_names=["Benign", "Malignant"]
)

# ------------------------------------------------------------
# (10) Manual Mean(|SHAP|) Table (Verification)
# ------------------------------------------------------------
mean_abs_shap = np.abs(shap_values[1]).mean(axis=0)

shap_df = (
    pd.DataFrame({
        "Feature": X.columns,
        "Mean(|SHAP|)": mean_abs_shap
    })
    .sort_values("Mean(|SHAP|)", ascending=False)
)

print(shap_df.head(10))
